# Carga de contribuyentes

Ingesta diaria del padron de contribuyentes hacia la capa silver.

## 1. Cabecera
> **Descripcion:** Informacion general del proceso: objetivo, version, responsable y tablas involucradas.

In [ ]:
# -------------------------------------------------------------------------
# PROYECTO       : Satelites - SUNAT
# PROCESO        : ETL_CONTRIBUYENTES
# OBJETIVO       : Actualizar el padron de contribuyentes
# VERSION        : 1.0.0
# DESARROLLADOR  : Eduardo Fajardo
# FECHA          : 28/08/2026
# TABLA FUENTE   : mb_bronze_prod.sat.gt_contribuyente
# TABLA DESTINO  : mb_silver_prod.sat.m_contribuyente
# FRECUENCIA     : Diaria
# -------------------------------------------------------------------------

## 2. Importacion de librerias
> **Descripcion:** Librerias estandar, de terceros y locales, en ese orden.

In [ ]:
import logging
import time
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 3. Lectura de parametros
> **Descripcion:** Parametros de ejecucion recibidos por widgets, para que el mismo codigo corra en cualquier ambiente.

In [ ]:
dbutils.widgets.text("p_fecha_carga", "")
dbutils.widgets.text("p_catalogo", "")

var_fecha_proceso = dbutils.widgets.get("p_fecha_carga")
var_catalogo = dbutils.widgets.get("p_catalogo")

logger = logging.getLogger("ETL_CONTRIBUYENTES")
logger.setLevel(logging.INFO)

ini_proceso = time.perf_counter()
logger.info("Inicio del proceso ETL_CONTRIBUYENTES")
logger.info("Parametros recibidos por widgets: fecha=%s catalogo=%s",
            var_fecha_proceso, var_catalogo)

## 4. Seccion constantes
> **Descripcion:** Valores que se mantienen constantes a lo largo del proceso.

In [ ]:
TBL_CONTRIBUYENTE_ORIGEN = f"{var_catalogo}.sat.gt_contribuyente"
TBL_CONTRIBUYENTE_FINAL = f"{var_catalogo}.sat.m_contribuyente"

EST_ACTIVO = "ACTIVO"

## 5. Funciones de transformacion
> **Descripcion:** Funciones modularizadas de lectura, transformacion y escritura.

In [ ]:
def read_contribuyente(tabla, fecha):
    """Lee el padron del dia proyectando solo las columnas requeridas."""
    return (
        spark.table(tabla)
        .select("nro_ruc", "nom_razon_social", "est_contribuyente", "fec_carga")
        .filter(F.col("fec_carga") == fecha)
    )


def add_razon_social_normalizada(df_origen):
    """Normaliza la razon social a mayusculas y sin espacios sobrantes."""
    return df_origen.withColumn(
        "nom_razon_social",
        F.upper(F.trim(F.col("nom_razon_social")))
    )


def validate_contribuyente_activo(df_origen):
    """Conserva unicamente los contribuyentes en estado activo."""
    return df_origen.filter(F.col("est_contribuyente") == EST_ACTIVO)

## 6. Logica del proceso
> **Descripcion:** Orquestacion de las funciones definidas previamente.

In [ ]:
ini_etapa = time.perf_counter()

df_contribuyente = read_contribuyente(TBL_CONTRIBUYENTE_ORIGEN, var_fecha_proceso)
df_contribuyente_activo = (
    df_contribuyente
    .transform(add_razon_social_normalizada)
    .transform(validate_contribuyente_activo)
)

logger.info("Tiempo de transformacion: %.2f segundos",
            time.perf_counter() - ini_etapa)

## 7. Deduplicacion
> **Descripcion:** Logica de deduplicacion segun las llaves de la tabla.

In [ ]:
ventana_ruc = Window.partitionBy("nro_ruc").orderBy(F.col("fec_carga").desc())

df_contribuyente_unico = (
    df_contribuyente_activo
    .withColumn("nro_orden", F.row_number().over(ventana_ruc))
    .filter(F.col("nro_orden") == 1)
    .drop("nro_orden")
)

## 8. Escritura en la tabla final
> **Descripcion:** Persistencia del resultado en formato Delta.

In [ ]:
ini_escritura = time.perf_counter()

try:
    (
        df_contribuyente_unico
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TBL_CONTRIBUYENTE_FINAL)
    )
except Exception as exc:
    logger.error("Fallo la escritura en %s: %s", TBL_CONTRIBUYENTE_FINAL, exc)
    raise

logger.info("Tiempo de escritura: %.2f segundos",
            time.perf_counter() - ini_escritura)

## 9. Registro de la ejecucion
> **Descripcion:** Cierre del proceso con el registro de duracion y volumen.

In [ ]:
logger.info("Fin del proceso ETL_CONTRIBUYENTES. Duracion total: %.2f segundos",
            time.perf_counter() - ini_proceso)